<div class="title-slide">

# Make the Most of What You’ve Got: Deep Learning Optimization with PyTorch Profiler and Holistic Trace Analysis 

## Collin Wilson

</div>

# Getting started with this demo

### This talk is given in a Jupyter notebook format. You can download it and all the materials to recreate the demo on GitHub:

```
git clone https://github.com/cwilson51/GIS-PP-and-HTA.git
```



# What we'll cover today

- ## PyTorch Profiler revisited
    - ### Inline profiling
    - ### Generating traces
- ## Examining traces and simple optimizations
- ## Holistic trace analysis (HTA)
    - ### Single GPU
    - ### Distributed workloads


# Working example - CPU

## ResNet50 on random inputs

In [3]:
import torch
from torchvision.models import resnet50
from torchvision import transforms
from torch.profiler import profile, record_function, ProfilerActivity, schedule

model = resnet50()
model.cuda()

inputs = torch.randn(1, 3, 224, 224)
labels = torch.randn(1, 1000)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

USDT:2026-11-13 13:24:47 561570:561570 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-13 13:24:47 561570:561570 SyncActivityProfilerHandler.cpp:46] profiler_stop


In [ ]:
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:

    model.train()
    with record_function("model_training"):
        outputs = model(inputs.cuda())
        loss = loss_fn(outputs, labels.cuda())
        loss.backward()
        optimizer.step()
        
    model.eval()
    with record_function("model_inference"):
        model(inputs.cuda())

In [5]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         model_training        29.31%      15.552ms        60.26%      31.977ms      31.977ms       0.000us         0.00%       5.939ms       5.939ms             1  
                                             cudaMalloc        26.01%      13.802ms        26.01%      13.802ms     394.353us       0.000us         0.00%       0.000us       0.000us            35  
         